# SalesPulse – Exploratory Data Analysis & Business Intelligence Workbook

**Product:** SalesPulse – Sales & Business Performance Intelligence  
**Tagline:** Transform Sales Data Into Smarter Business Decisions  
**Dataset:** Sample Superstore Sales Dataset (9,994 transaction records, 2014-2017)  
**Objective:** Analyze sales velocity, operating profit margins, discount elasticity, regional performance, and customer cohorts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Paths
DATA_PATH = Path('../data/processed/clean_superstore.csv')
df = pd.read_csv(DATA_PATH)
df['Order Date'] = pd.to_datetime(df['Order Date'])
print(f"Loaded {len(df):,} records with {len(df.columns)} columns.")
df.head(3)

## 1. High-Level Financial Baseline & Integrity Check
Verifying total revenue, profit, quantity, and overall operating margin.

In [ ]:
total_revenue = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = df['Order ID'].nunique()
total_units = df['Quantity'].sum()
overall_margin = (total_profit / total_revenue) * 100
loss_transactions = (df['Profit'] < 0).sum()
total_losses = abs(df[df['Profit'] < 0]['Profit'].sum())

print("=" * 45)
print(f"Total Revenue:           ${total_revenue:,.2f}")
print(f"Total Net Profit:        ${total_profit:,.2f}")
print(f"Overall Profit Margin:   {overall_margin:.2f}%")
print(f"Distinct Orders:         {total_orders:,}")
print(f"Total Quantity Sold:     {total_units:,} units")
print(f"Average Order Value:     ${(total_revenue / total_orders):.2f}")
print(f"Loss Transactions:       {loss_transactions:,} ({loss_transactions/len(df)*100:.1f}%)")
print(f"Cumulative Loss Drag:    -${total_losses:,.2f}")
print("=" * 45)

## 2. Temporal Analysis: Monthly & Annual YoY Revenue Trajectory

In [ ]:
monthly = df.groupby('Order Year Month').agg(
    revenue=('Sales', 'sum'),
    profit=('Profit', 'sum')
).reset_index()

plt.figure(figsize=(14, 5))
plt.plot(monthly['Order Year Month'], monthly['revenue'], marker='o', label='Revenue ($)', color='#6366F1', linewidth=2)
plt.plot(monthly['Order Year Month'], monthly['profit'], marker='s', label='Net Profit ($)', color='#10B981', linewidth=2)
plt.xticks(rotation=90, fontsize=8)
plt.title('SalesPulse: Monthly Revenue & Profit Trajectory (2014-2017)', fontsize=14, fontweight='bold')
plt.ylabel('USD ($)')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Product Category & Sub-Category Profitability Diagnostics
Examining the margin disparity between Technology, Office Supplies, and Furniture.

In [ ]:
cat_summary = df.groupby('Category').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Units=('Quantity', 'sum')
).reset_index()
cat_summary['Margin %'] = (cat_summary['Profit'] / cat_summary['Revenue']) * 100
display(cat_summary.sort_values('Revenue', ascending=False))

# Sub-category profit distribution
subcat = df.groupby(['Category', 'Sub-Category']).agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', lambda x: round(x.mean() * 100, 1))
).reset_index()
subcat['Margin %'] = ((subcat['Profit'] / subcat['Revenue']) * 100).round(2)
display(subcat.sort_values('Profit', ascending=True).head(5))

## 4. The Discounting Trap: Price Elasticity vs Margin Erosion
Demonstrating the empirical collapse in margins when discounts exceed 20%.

In [ ]:
tier_summary = df.groupby('Discount Tier').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Transactions=('Row ID', 'count'),
    Loss_Transactions=('Profit', lambda x: (x < 0).sum())
).reset_index()

tier_summary['Profit Margin %'] = ((tier_summary['Profit'] / tier_summary['Revenue']) * 100).round(2)
tier_summary['Loss Rate %'] = ((tier_summary['Loss_Transactions'] / tier_summary['Transactions']) * 100).round(2)
display(tier_summary)

corr = df['Discount'].corr(df['Profit'])
print(f"Pearson Correlation (Discount vs Profit): {corr:.4f}")

## 5. Regional & Territorial Performance Analysis

In [ ]:
regional = df.groupby('Region').agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Orders=('Order ID', 'nunique')
).reset_index()
regional['Margin %'] = ((regional['Profit'] / regional['Revenue']) * 100).round(2)
regional['Revenue Share %'] = ((regional['Revenue'] / total_revenue) * 100).round(2)
display(regional.sort_values('Revenue', ascending=False))

# Bottom 5 loss-making states
loss_states = df.groupby(['Region', 'State']).agg(
    Revenue=('Sales', 'sum'),
    Profit=('Profit', 'sum'),
    Avg_Discount=('Discount', lambda x: round(x.mean() * 100, 1))
).reset_index()
loss_states = loss_states[loss_states['Profit'] < 0].sort_values('Profit', ascending=True)
print("Worst Loss-Making States:")
display(loss_states.head(5))

## 6. Pareto Analysis (80/20 Customer Concentration)

In [ ]:
cust = df.groupby('Customer ID')['Sales'].sum().sort_values(ascending=False).reset_index()
cust['Cumulative_Sales'] = cust['Sales'].cumsum()
cust['Cumulative_Pct'] = (cust['Cumulative_Sales'] / total_revenue) * 100
cust['Customer_Rank'] = np.arange(1, len(cust) + 1)
cust['Customer_Pct'] = (cust['Customer_Rank'] / len(cust)) * 100

p80 = cust[cust['Cumulative_Pct'] >= 80.0].iloc[0]
print(f"{p80['Customer_Pct']:.1f}% of customers ({int(p80['Customer_Rank'])} accounts) account for 80% of total revenue.")